# Lesson 00: Check your setup

Run every cell from top to bottom (**Kernel → Restart & Run All**).

It checks:
1. Python, PyTorch and OpenCV are installed
2. PyTorch can see your GPU (CUDA)
3. Your COD dataset folder exists, and what is inside it

👉 **Send me the output of section 3.** The folder layout tells me how to write the dataset loader in a later lesson.

> The dataset path is set once in `config.py` (`DATASET_ROOT`). Every notebook in this folder imports it.

In [ ]:
import os
import sys

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch

from config import DATASET_ROOT

## 1) Libraries

In [ ]:
print(f"Python  : {sys.version.split()[0]}")
print(f"PyTorch : {torch.__version__}")
print(f"OpenCV  : {cv2.__version__}")
print(f"NumPy   : {np.__version__}")

## 2) GPU

If this says CUDA is **not** available but you have an NVIDIA GPU, you installed the CPU-only PyTorch.
Reinstall it with the command in `README.md`.

In [ ]:
if not torch.cuda.is_available():
    print("CUDA is NOT available: PyTorch will use the CPU (slow for training).")
else:
    print(f"CUDA version (PyTorch build): {torch.version.cuda}")
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"GPU {i}: {props.name}  |  memory: {props.total_memory / 1024**3:.1f} GB")

    # A tiny test: multiply two matrices on the GPU.
    a = torch.randn(1000, 1000, device="cuda")
    b = torch.randn(1000, 1000, device="cuda")
    c = a @ b
    print(f"GPU test OK: result shape {tuple(c.shape)} on {c.device}")

## 3) Dataset layout

Prints the folder tree (3 levels deep) with the number of files in each folder and one example file name.

In [ ]:
print(f"DATASET_ROOT = {DATASET_ROOT}")
if not os.path.isdir(DATASET_ROOT):
    print("Folder NOT found. Fix DATASET_ROOT in config.py.")
else:
    base_depth = DATASET_ROOT.rstrip("\\/").count(os.sep)
    for dirpath, dirnames, filenames in os.walk(DATASET_ROOT):
        depth = dirpath.count(os.sep) - base_depth
        if depth > 3:
            dirnames[:] = []  # don't go deeper
            continue
        dirnames.sort()
        indent = "    " * depth
        name = os.path.basename(dirpath) or dirpath
        example = f"  e.g. {sorted(filenames)[0]}" if filenames else ""
        print(f"{indent}{name}/  ({len(filenames)} files){example}")

## 4) Look at one sample

Finds the first image and a mask with the same name in a neighbouring `GT` / `mask` folder,
then shows **image | mask | overlay** (the camouflaged object painted red).

In [ ]:
def show(images, titles=None, cmap=None, cols=4, size=4):
    """Show one or more images inline. Accepts OpenCV BGR (3-channel) or grayscale arrays."""
    if not isinstance(images, (list, tuple)):
        images = [images]
    titles = titles or [""] * len(images)
    rows = (len(images) + cols - 1) // cols
    cols = min(cols, len(images))
    fig, axes = plt.subplots(rows, cols, figsize=(size * cols, size * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, img, t in zip(axes.flat, images, titles):
        if img.ndim == 3:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)  # OpenCV is BGR, matplotlib expects RGB
        ax.imshow(img, cmap=cmap or ("gray" if img.ndim == 2 else None))
        ax.set_title(t)
    plt.tight_layout()
    plt.show()

In [ ]:
image_path, mask_path = None, None
if os.path.isdir(DATASET_ROOT):
    for dirpath, _, filenames in os.walk(DATASET_ROOT):
        folder = dirpath.lower()
        if "imgs" in folder or "image" in folder:
            jpgs = sorted(f for f in filenames if f.lower().endswith((".jpg", ".png")))
            if jpgs:
                image_path = os.path.join(dirpath, jpgs[0])
                stem = os.path.splitext(jpgs[0])[0]
                parent = os.path.dirname(dirpath)
                for sibling in sorted(os.listdir(parent)):
                    if "gt" in sibling.lower() or "mask" in sibling.lower():
                        candidate = os.path.join(parent, sibling, stem + ".png")
                        if os.path.isfile(candidate):
                            mask_path = candidate
                            break
                break

print(f"Sample image: {image_path}")
print(f"Sample mask : {mask_path}")

if image_path is not None:
    image = cv2.imread(image_path)
    panels, names = [image], ["image"]
    if mask_path:
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.resize(mask, (image.shape[1], image.shape[0]))
        overlay = image.copy()
        red = np.array([0, 0, 255])  # BGR
        overlay[mask > 127] = (0.5 * overlay[mask > 127] + 0.5 * red).astype(np.uint8)
        panels += [mask, overlay]
        names += ["mask (ground truth)", "overlay"]
    show(panels, names, cols=3)
else:
    print("Could not find an image folder automatically. Send me the tree printed above.")